In [1]:
import sys, json, os
from pathlib import Path

# Racine du projet = dossier parent du dossier "notebooks"
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))          # pour importer src.processing.*
sys.path.insert(0, str(PROJECT_ROOT / "dags")) # pour importer utils.s3_client

from src.processing.clean_passes import extract_clean_passes
from src.processing.aggregate_networks import aggregate_pass_network, write_silver

MATCH_ID = 3869685
EVENTS_PATH = PROJECT_ROOT / "data" / "statsbomb" / "data" / "events" / f"{MATCH_ID}.json"

print("Events file existe :", EVENTS_PATH.exists())

Events file existe : True


In [2]:
# 1) Charger le JSON brut des events du match
with open(EVENTS_PATH, encoding="utf-8") as f:
    events = json.load(f)

print(f"Total events dans le match : {len(events)}")

# 2) Filtrer -> passes réussies uniquement
clean = extract_clean_passes(events, MATCH_ID)
print(f"Passes réussies extraites : {len(clean)}")

# 3) Aperçu des 5 premières
clean[:5]

Total events dans le match : 4407
Passes réussies extraites : 994


[{'passeur': 'Antoine Griezmann',
  'receveur': 'Aurélien Djani Tchouaméni',
  'equipe': 'France',
  'match_id': 3869685},
 {'passeur': 'Nahuel Molina Lucero',
  'receveur': 'Rodrigo Javier De Paul',
  'equipe': 'Argentina',
  'match_id': 3869685},
 {'passeur': 'Rodrigo Javier De Paul',
  'receveur': 'Cristian Gabriel Romero',
  'equipe': 'Argentina',
  'match_id': 3869685},
 {'passeur': 'Cristian Gabriel Romero',
  'receveur': 'Nicolás Hernán Otamendi',
  'equipe': 'Argentina',
  'match_id': 3869685},
 {'passeur': 'Nicolás Hernán Otamendi',
  'receveur': 'Cristian Gabriel Romero',
  'equipe': 'Argentina',
  'match_id': 3869685}]

In [3]:
# Agréger : compter combien de fois chaque paire (passeur -> receveur) s'est répétée
agg = aggregate_pass_network(clean)

print(f"Nombre d'arêtes uniques du réseau : {len(agg)}")
print(f"Total passes reconstitué : {agg['nb'].sum()}  (doit égaler {len(clean)})")
print()
print("Top 10 des connexions les plus actives :")
agg.head(10)

Nombre d'arêtes uniques du réseau : 272
Total passes reconstitué : 994  (doit égaler 994)

Top 10 des connexions les plus actives :


,passeur,receveur,equipe,match_id,nb
100,Nicolás Hernán Otamendi,Cristian Gabriel Romero,Argentina,3869685,18
248,Raphaël Varane,Dayotchanculle Upamecano,France,3869685,16
92,Nahuel Molina Lucero,Rodrigo Javier De Paul,Argentina,3869685,15
107,Nicolás Hernán Otamendi,Nicolás Alejandro Tagliafico,Argentina,3869685,15
207,Jules Koundé,Raphaël Varane,France,3869685,15
102,Nicolás Hernán Otamendi,Enzo Fernandez,Argentina,3869685,14
12,Cristian Gabriel Romero,Enzo Fernandez,Argentina,3869685,14
117,Rodrigo Javier De Paul,Lionel Andrés Messi Cuccittini,Argentina,3869685,14
173,Dayotchanculle Upamecano,Raphaël Varane,France,3869685,14
39,Enzo Fernandez,Rodrigo Javier De Paul,Argentina,3869685,14


In [4]:
import duckdb

# Créer le dossier de sortie
silver_dir = PROJECT_ROOT / "data" / "silver"
silver_dir.mkdir(parents=True, exist_ok=True)

dest = silver_dir / "passes_agg.parquet"

# Écrire le Parquet via DuckDB
con = duckdb.connect()
write_silver(con, agg, str(dest))
con.close()

print(f"Parquet écrit : {dest}")
print(f"Taille : {dest.stat().st_size / 1024:.1f} KB")

Parquet écrit : C:\Users\fredf\pitch-intelligence-graphs\data\silver\passes_agg.parquet
Taille : 3.5 KB


In [5]:
# Relire le Parquet -> s'assurer qu'il est bien lisible avec la structure attendue
verif = duckdb.connect().execute(f"SELECT * FROM read_parquet('{dest}') LIMIT 5").df()
print("Colonnes :", list(verif.columns))
print("Types :")
print(verif.dtypes)
verif

Colonnes : ['passeur', 'receveur', 'equipe', 'match_id', 'nb']
Types :
passeur     object
receveur    object
equipe      object
match_id     int64
nb           int64
dtype: object


,passeur,receveur,equipe,match_id,nb
0,Nicolás Hernán Otamendi,Cristian Gabriel Romero,Argentina,3869685,18
1,Raphaël Varane,Dayotchanculle Upamecano,France,3869685,16
2,Nahuel Molina Lucero,Rodrigo Javier De Paul,Argentina,3869685,15
3,Nicolás Hernán Otamendi,Nicolás Alejandro Tagliafico,Argentina,3869685,15
4,Jules Koundé,Raphaël Varane,France,3869685,15
